In [1]:
print("all ok")

all ok


In [2]:
import uuid
from dataclasses import dataclass
from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)
from langgraph.runtime import Runtime

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0
)

e:\AI-Learning\Gen-AI\Gen_AI_Practice\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [5]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

In [6]:
@dataclass
class Context:
    user_id: str


In [7]:
Context.user_id=str(uuid.uuid4())

In [8]:
Context.user_id

'd3536bf6-4e55-4c75-a778-063a3b3268d5'

In [23]:
def assistant_node(
    state: MessagesState,
    runtime: Runtime[Context]
):
    print("runtime",runtime)
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "memories")
    user_message = state["messages"][-1].content
    
    if user_message.lower().startswith("remember:"):
        memory = user_message[len("remember:"):].strip()
        runtime.store.put(
            namespace,
            str(uuid.uuid4()),
            {
                "data": memory
            }
        )
        print(
            f"[MEMORY SAVED]: {memory}"
        )
        
    memories = runtime.store.search(namespace)
    memory_text = "\n".join(
        memory.value["data"]
        for memory in memories
    )
    if not memory_text:
        memory_text = "No saved memories."
        
    
    system_prompt = f"""
    You are a helpful assistant.

    Long-term memories about this user:

    {memory_text}
    """


    response = model.invoke(
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *state["messages"]
        ]
    )
    return {
        "messages": [response]
    }

In [24]:
builder = StateGraph(
    MessagesState,
    context_schema=Context
)


In [25]:
builder.add_node(
    "assistant",
    assistant_node
)

In [26]:
builder.add_edge(
    START,
    "assistant"
)

builder.add_edge(
    "assistant",
    END
)

In [42]:
graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [43]:
def chat(
    message,
    user_id,
    thread_id
):

    result = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },

        config={
            "configurable": {
                "thread_id": thread_id
            }
        },

        context=Context(
            user_id=user_id
        )
    )


    answer = result["messages"][-1].content

    print("\nUSER:")
    print(message)

    print("\nASSISTANT:")
    print(answer)

    print("\n" + "=" * 60)

In [52]:
chat(
    message="Remember: My current project is Agentic RAG.",
    user_id="sunny_123",
    thread_id="thread_1"
)

runtime Runtime(context=Context(user_id='sunny_123'), store=<langgraph.store.memory.InMemoryStore object at 0x0000018335D1A030>, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000018350D86CA0>, heartbeat=<function _no_op_heartbeat at 0x0000018335C6A020>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1d9c29-ba16-6ce8-8012-d05866c73c7d', checkpoint_ns='assistant:4160688a-ea59-cbb4-e24d-4f3ca458055c', task_id='4160688a-ea59-cbb4-e24d-4f3ca458055c', thread_id='thread_1', run_id=None, node_attempt=1, node_first_attempt_time=1793956528.3126504), server_info=None, control=<langgraph.runtime.RunControl object at 0x000001835016FBB0>)
[MEMORY SAVED]: My current project is Agentic RAG.

USER:
Remember: My current project is Agentic RAG.

ASSISTANT:
Got it! Your current project is Agentic RAG, and your favorite programming language is Python. If you need any assistance related to Agentic RAG or Python, just let me know! How can I help you today?



In [53]:
chat(
    message="What is my current project?",
    user_id="sunny_123",
    thread_id="thread_2"
)

runtime Runtime(context=Context(user_id='sunny_123'), store=<langgraph.store.memory.InMemoryStore object at 0x0000018335D1A030>, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000018350D87560>, heartbeat=<function _no_op_heartbeat at 0x0000018335C6A020>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1d9c29-e068-69c6-8009-0c51edc18a8f', checkpoint_ns='assistant:d532065d-20d2-1182-047e-59b0e7d7dc36', task_id='d532065d-20d2-1182-047e-59b0e7d7dc36', thread_id='thread_2', run_id=None, node_attempt=1, node_first_attempt_time=1793956532.331803), server_info=None, control=<langgraph.runtime.RunControl object at 0x00000183502742B0>)

USER:
What is my current project?

ASSISTANT:
Your current project is Agentic RAG.



In [30]:
chat(
    message="Remember: my favorite programming language is Python.",
    user_id="sunny_123",
    thread_id="thread_1"
)

runtime Runtime(context=Context(user_id='sunny_123'), store=<langgraph.store.memory.InMemoryStore object at 0x0000018335D1A030>, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x000001834DE9F6A0>, heartbeat=<function _no_op_heartbeat at 0x0000018335C6A020>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1d9bf7-ccec-6d63-8009-cc3d8a987bd8', checkpoint_ns='assistant:09088d41-dac6-6eb8-6891-841bd09acb7c', task_id='09088d41-dac6-6eb8-6891-841bd09acb7c', thread_id='thread_1', run_id=None, node_attempt=1, node_first_attempt_time=1793955188.1145935), server_info=None, control=<langgraph.runtime.RunControl object at 0x000001834FD60100>)
[MEMORY SAVED]: my favorite programming language is Python.

USER:
Remember: my favorite programming language is Python.

ASSISTANT:
Got it! Your favorite programming language is Python. If you need any help with your Agentic RAG project using Python, just let me know! How can I assist you further?



In [31]:
chat(
    message="What is my favorite programming language?",
    user_id="sunny_123",
    thread_id="thread_2"
)

runtime Runtime(context=Context(user_id='sunny_123'), store=<langgraph.store.memory.InMemoryStore object at 0x0000018335D1A030>, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x000001834DE9F6A0>, heartbeat=<function _no_op_heartbeat at 0x0000018335C6A020>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1d9bf8-3249-655a-8000-d1d296fb9711', checkpoint_ns='assistant:ad3835de-18c7-4178-06ee-bfb77d36a698', task_id='ad3835de-18c7-4178-06ee-bfb77d36a698', thread_id='thread_2', run_id=None, node_attempt=1, node_first_attempt_time=1793955198.7389786), server_info=None, control=<langgraph.runtime.RunControl object at 0x000001834FD60100>)

USER:
What is my favorite programming language?

ASSISTANT:
Your favorite programming language is Python.



                 sunny_123
                     ↓
              InMemoryStore
                     ↓
      "favorite language = Python"
                     ↓
             ┌───────┴───────┐
             ↓               ↓
         thread_1         thread_2

In [41]:
cfg = {"configurable": {"thread_id": "1"}}

graph.invoke(
    {"messages": [("user", "Hi, I'm Srinjay")]},
    cfg,
    context=Context(user_id="1"),
)


runtime Runtime(context=Context(user_id='1'), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000018350D32CA0>, heartbeat=<function _no_op_heartbeat at 0x0000018335C6A020>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1d9c0e-562f-68c9-800e-2034fd6fc5fd', checkpoint_ns='assistant:de844f55-22d4-054a-3658-42fc97c64709', task_id='de844f55-22d4-054a-3658-42fc97c64709', thread_id='1', run_id=None, node_attempt=1, node_first_attempt_time=1793955793.0628505), server_info=None, control=<langgraph.runtime.RunControl object at 0x0000018350275DB0>)


AttributeError: 'NoneType' object has no attribute 'search'

In [36]:
graph.invoke(
    {"messages": [("user", "What's my name?")]},
    cfg,
    context=Context(user_id="2"),
)

runtime Runtime(context=Context(user_id='2'), store=<langgraph.store.memory.InMemoryStore object at 0x0000018335D1A030>, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000018350C7BBA0>, heartbeat=<function _no_op_heartbeat at 0x0000018335C6A020>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1d9c08-2095-61e2-8009-4669c4845a1b', checkpoint_ns='assistant:1a4c2073-6f1b-5035-c126-af366f35d1da', task_id='1a4c2073-6f1b-5035-c126-af366f35d1da', thread_id='1', run_id=None, node_attempt=1, node_first_attempt_time=1793955626.3809884), server_info=None, control=<langgraph.runtime.RunControl object at 0x0000018350160610>)


{'messages': [HumanMessage(content="Hi, I'm Srinjay", additional_kwargs={}, response_metadata={}, id='b0843c8d-628b-44f8-92b6-431031b8b810'),
  HumanMessage(content="Hi, I'm Srinjay", additional_kwargs={}, response_metadata={}, id='44b61424-8115-4950-889a-a842ec7619e2'),
  HumanMessage(content="Hi, I'm Srinjay", additional_kwargs={}, response_metadata={}, id='d7af79e7-734f-49eb-8435-6cd6ac5a2337'),
  HumanMessage(content="Hi, I'm Srinjay", additional_kwargs={}, response_metadata={}, id='a6990fa8-efc2-4a40-b58c-bfb03018a964'),
  AIMessage(content='Hello Srinjay! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-20

In [40]:
chat(
    message="Remember: my favorite language is Python.",
    user_id="sunny_123",
    thread_id="sunny_thread_1"
)

[MEMORY SAVED]: my favorite language is Python.

USER:
Remember: my favorite language is Python.

ASSISTANT:
Got it! I've remembered that your favorite language is Python. If there's anything related to Python you'd like help with, just let me know!



In [41]:
chat(
    message="Remember: my favorite language is Java.",
    user_id="rahul_456",
    thread_id="rahul_thread_1"
)

[MEMORY SAVED]: my favorite language is Java.

USER:
Remember: my favorite language is Java.

ASSISTANT:
Got it! I've remembered that your favorite language is Java. If you need any help with Java or anything else, just let me know!



users
│
├── sunny_123
│      └── memories
│           └── favorite language = Python
│
└── rahul_456
       └── memories
            └── favorite language = Java

In [42]:
chat(
    message="What is my favorite programming language?",
    user_id="rahul_456",
    thread_id="rahul_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Your favorite programming language is Java.



In [43]:
chat(
    message="What is my favorite programming language?",
    user_id="sunny_123",
    thread_id="sunny_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Your favorite programming language is Python.



In [46]:
user_id = "sunny_123"

In [47]:
namespace = (
    "users",
    user_id,
    "memories"
)


In [48]:
memories = store.search(namespace)

for memory in memories:
    print(memory.value)

{'data': 'my favorite programming language is Python.'}


In [47]:
memories = store.search(
    ("users", "sunny_123", "memories")
)

for memory in memories:
    print(memory.value["data"])

my favorite programming language is Python.
my favorite language is Python.
